# Investigación de datos de "base.xlsx"

Columnas Criticas
**`Periodo`**: idetnificar y normalizar las fechas.                  
**`Neto`**: (negativos, ceros y atípicos) y la "integridad referencial".

In [47]:
import re
import sys
from collections import Counter
from datetime import date, datetime
from pathlib import Path
import openpyxl
import pandas as pd

RAIZ = Path.cwd()
if not (RAIZ / "Base.xlsx").exists():
    RAIZ = RAIZ.parent

pd.set_option("display.width", 120)
ARCHIVO = RAIZ / "Base.xlsx"
ventas = pd.read_excel(ARCHIVO, sheet_name="Ventas", dtype=str)   
print(f"{len(ventas):,} filas de ventas cargadas")

446,742 filas de ventas cargadas


### validación de diferentes tipos de datos en la columna Periodo
bloque para conocer las diferencias y cantidades

In [48]:
for x, cantidad in ventas["Periodo"].dropna().astype(str).value_counts().sort_index().items():
    print(f"{repr(x)} -> {cantidad}")

'2026 06' -> 70305
'2026-04' -> 70867
'2026.01' -> 82387
'2026.03' -> 79892
'2026/05' -> 72509
'2026_02' -> 70782


### Identificando las diferencias normalicemos todas las fechas:

In [49]:
_PERIODO = re.compile(r"(\d{4})\D+?(\d{1,2})")


def norm_periodo_local(valor):
    if valor is None:
        return None
    if isinstance(valor, (datetime, date)):
        return f"{valor.year:04d}-{valor.month:02d}"
    m = _PERIODO.search(str(valor))
    if not m:
        return None
    year, mes = int(m.group(1)), int(m.group(2))
    return f"{year:04d}-{mes:02d}" if 1 <= mes <= 12 else None


ventas["periodo"] = ventas["Periodo"].map(norm_periodo_local)
print(ventas["periodo"].value_counts().sort_index().to_string())
print("periodos:", ventas['periodo'].nunique(), "| nulos:", int(ventas['periodo'].isna().sum()))

periodo
2026-01    82387
2026-02    70782
2026-03    79892
2026-04    70867
2026-05    72509
2026-06    70305
periodos: 6 | nulos: 0


### validación de los tipos de datos en la columna Cod Principal
al leer con dtype=str pandas esconde que hay celdas numéricas y celdas de texto con espacios.
para verlo hay que leer el archivo crudo con openpyxl


In [50]:
hoja_cruda = openpyxl.load_workbook(ARCHIVO, read_only=True)["Ventas"]

tipos = Counter(
    type(codigo).__name__
    for i, (codigo,) in enumerate(
        hoja_cruda.iter_rows(min_row=2, min_col=2, max_col=2, values_only=True)
    )
    if i < 50_000
)

print("Tipos de 'Cod Principal':", dict(tipos))

crudo = ventas["Cod Principal"]

padding = crudo[crudo != crudo.str.strip()]
print(
    f"Texto con padding: {len(padding)} -> "
    f"{[repr(x) for x in padding.unique()[:3]]}"
)



Tipos de 'Cod Principal': {'str': 46, 'int': 49954}
Texto con padding: 305 -> ["'  1000001'"]


#### Cambiando a INT

In [51]:
ventas["cliente"] = crudo.map(to_int_local)
print("No numéricos:", ventas["cliente"].isna().sum())
print("Clientes distintos:", ventas["cliente"].nunique())

No numéricos: 0
Clientes distintos: 11288


### validación de duplicados en las tablas maestras
Revición de valores Duplicados en "Cod Cliente", "Cod Material", "Cod Asesor"

In [52]:
for hoja, llaves in [
    ("Clientes", ["Cod Cliente"]),
    ("Materiales", ["Cod Material"]),
    ("Asesores", ["Cod Cliente", "Cod Asesor"]),
]:
    datos = pd.read_excel(ARCHIVO, sheet_name=hoja, dtype=str)
    columnas_datos = [col for col in datos.columns if col not in llaves]
    duplicados = datos.duplicated(subset=llaves).sum()
    conflictos = (
        datos.groupby(llaves)[columnas_datos]
        .nunique()
        .max(axis=1)
        .gt(1)
        .sum()
    )
    filas_unicas = datos.drop_duplicates(subset=llaves)
    print(
        f"{hoja:11} | "
        f"filas: {len(datos):>7,} → {len(filas_unicas):>6,} | "
        f"duplicados: {duplicados:>4} | "
        f"conflictos: {conflictos}"
    )


# Mostrar un ejemplo de duplicado
clientes = pd.read_excel(ARCHIVO, sheet_name="Clientes", dtype=str)

duplicados = clientes[clientes["Cod Cliente"].duplicated(keep=False)]

if not duplicados.empty:
    codigo_duplicado = duplicados["Cod Cliente"].iloc[0]

    print("\nEjemplo de duplicado:")
    print(
        clientes[clientes["Cod Cliente"] == codigo_duplicado]
        .to_string(index=False)
    )

Clientes    | filas:  11,421 → 11,288 | duplicados:  133 | conflictos: 0
Materiales  | filas:   2,888 →  2,794 | duplicados:   94 | conflictos: 0
Asesores    | filas:  11,620 → 11,288 | duplicados:  332 | conflictos: 0

Ejemplo de duplicado:
Cod Cliente         Nombre Cliente Tipo Cliente
    1000001 Hotel La Alameda Norte        Hotel
    1000001 Hotel La Alameda Norte        Hotel


### validación de negativos, ceros y atípicos en la columna Neto
validar de donde vienen los valores negativos
¿Notas Credito?

In [53]:
neto = ventas["Neto"].astype(float)
ventas["neto"] = neto
ventas["negativo"] = neto < 0
ventas["cero"] = neto == 0

print(f"filas={len(neto):,}   negativos={int((neto < 0).sum()):,} "
      f"({neto[neto < 0].sum():,.2f})   ceros={int((neto == 0).sum()):,}")
print(f"atípicos: >1M={int((neto > 1e6).sum()):,}  >10M={int((neto > 1e7).sum()):,}  "
      f"p99={neto.abs().quantile(0.99):,.2f}  max={neto.max():,.2f}")

desglose = (ventas.groupby("periodo")
            .agg(filas=("neto", "size"), negativos=("negativo", "sum"), ceros=("cero", "sum"),
                 neto_total=("neto", "sum")))
desglose["pct_negativos"] = (desglose.negativos / desglose.filas * 100).round(2)
print("\npor periodo:")
print(desglose.round(2).to_string())
print("\nclientes distintos con nota crédito:", ventas.loc[ventas.negativo, "cliente"].nunique())
print("las 5 ventas más altas (cola larga, mismo cliente):")
print(ventas.nlargest(5, "neto")[["periodo", "cliente", "neto"]].to_string(index=False))

filas=446,742   negativos=3,127 (-123,618,789.14)   ceros=11,428
atípicos: >1M=2,078  >10M=103  p99=563,844.26  max=111,428,571.43

por periodo:
         filas  negativos  ceros    neto_total  pct_negativos
periodo                                                      
2026-01  82387        397   1490  3.182582e+09           0.48
2026-02  70782        303   2255  2.635681e+09           0.43
2026-03  79892        891   2233  3.651199e+09           1.12
2026-04  70867        540   1940  3.437598e+09           0.76
2026-05  72509        561   1853  3.397503e+09           0.77
2026-06  70305        435   1657  3.387903e+09           0.62

clientes distintos con nota crédito: 1283
las 5 ventas más altas (cola larga, mismo cliente):
periodo   cliente         neto
2026-03   1004017 1.114286e+08
2026-03   1004017 4.285714e+07
2026-02 700005314 3.713705e+07
2026-06   1004017 3.659568e+07
2026-06 700026935 3.589456e+07


### ¿las notas crédito están respaldadas por una factura?
revición de facturas que con el mismo cliente y con el mismo valor se realizo previamente con y se "Anulo" con una Nota Credito.

In [54]:
ventas["material"] = ventas["Cod Material"].map(to_int)
ventas["neto"] = ventas["neto"].round(2)

notas = ventas[ventas.neto < 0].copy()
notas["monto"] = -notas.neto
positivas = ventas[ventas.neto > 0]


# 1. ¿La nota crédito ocurre después de una venta?
pares = positivas.merge(
    notas[["periodo", "cliente", "material"]].rename(columns={"periodo": "periodo_nota"}),
    on=["cliente", "material"]
)
pares = pares[pares.periodo < pares.periodo_nota]

pares["meses"] = (
    pd.PeriodIndex(pares.periodo_nota, freq="M")
    - pd.PeriodIndex(pares.periodo, freq="M")
).map(lambda x: x.n)

desfase = pares.groupby(
    ["periodo_nota", "cliente", "material"]
)["meses"].min()

print(f"Notas con venta previa: {len(desfase):,} de {len(notas):,}")
print("Meses entre venta y nota:", desfase.value_counts().sort_index().to_dict())


# 2. ¿La nota corresponde al valor exacto de una venta del cliente?
ventas_cliente = positivas.groupby("cliente")["neto"].apply(set)

exactas = [
    monto in ventas_cliente.get(cliente, set())
    for cliente, monto in zip(notas.cliente, notas.monto)
]

print(
    f"Notas con valor idéntico a una venta: "
    f"{sum(exactas):,} ({sum(exactas) / len(notas) * 100:.1f}%)"
)


# 3. ¿Una venta anterior cubre el valor de la nota?
anteriores = positivas.merge(
    notas[["periodo", "cliente", "material", "monto"]]
    .rename(columns={"periodo": "periodo_nota"}),
    on=["cliente", "material"]
)

anteriores = anteriores[anteriores.periodo < anteriores.periodo_nota]

maximos = (
    anteriores
    .groupby(["periodo_nota", "cliente", "material"])["neto"]
    .max()
    .rename("maximo")
)

cubiertas = notas.join(
    maximos,
    on=["periodo", "cliente", "material"]
).fillna({"maximo": 0})

cubre = cubiertas.maximo >= cubiertas.monto

print(
    f"Notas cubiertas por una venta previa: "
    f"{cubre.sum():,} ({cubre.mean() * 100:.1f}%)"
)


# Control: misma prueba sobre 4.000 ventas positivas
muestra = positivas.sample(4000, random_state=7)

control = muestra.merge(
    positivas[["periodo", "cliente", "material"]]
    .rename(columns={"periodo": "periodo_venta"}),
    on=["cliente", "material"]
)

control = control[control.periodo_venta < control.periodo]

maximos_control = (
    control
    .groupby(["periodo", "cliente", "material"])["neto"]
    .max()
)

control = muestra.join(
    maximos_control,
    on=["periodo", "cliente", "material"],
    rsuffix="_max"
).fillna({"neto_max": 0})

print(
    f"CONTROL (4.000 ventas) cubiertas por una venta previa: "
    f"{(control.neto_max >= control.neto).mean() * 100:.1f}%"
)


# 4. Clientes con más notas crédito
print("\nNotas crédito por cliente (top 3):")
print(
    notas.groupby("cliente")["monto"]
    .sum()
    .nlargest(3)
    .round(0)
    .to_string()
)

Notas con venta previa: 2,264 de 3,127
Meses entre venta y nota: {1: 1892, 2: 249, 3: 87, 4: 32, 5: 4}
Notas con valor idéntico a una venta: 819 (26.2%)
Notas cubiertas por una venta previa: 2,235 (71.5%)
CONTROL (4.000 ventas) cubiertas por una venta previa: 64.3%

Notas crédito por cliente (top 3):
cliente
1000018      20131305.0
1005351      13574857.0
700025370    10880357.0


### ¿notas crédito sin ninguna factura que las respalde?
un cliente que solo devuelve (tiene notas crédito y ninguna compra) no está respaldado por
ninguna venta del período

In [55]:
solo_devuelve = (ventas.groupby("cliente")["neto"]
                 .agg(lambda columna: (columna > 0).sum() == 0 and (columna < 0).any()))
grupo = ventas[ventas.cliente.isin(solo_devuelve[solo_devuelve].index)]
notas_sin_respaldo = grupo[grupo.neto < 0]
total_notas = abs(ventas.loc[ventas.neto < 0, "neto"].sum())

print(f"clientes que solo devuelven: {len(solo_devuelve[solo_devuelve])}  "
      f"(filas en cero de ese grupo: {int((grupo.neto == 0).sum())})")
print(f"notas sin factura que las respalde: {len(notas_sin_respaldo)} por "
      f"{abs(notas_sin_respaldo.neto.sum()):,.0f} "
      f"({abs(notas_sin_respaldo.neto.sum())/total_notas*100:.2f}% de las notas crédito)")
print("por mes:", notas_sin_respaldo.periodo.value_counts().sort_index().to_dict())
print("clientes que cierran el período con saldo negativo:",
      int((ventas.groupby("cliente")["neto"].sum() < 0).sum()))

clientes que solo devuelven: 36  (filas en cero de ese grupo: 42)
notas sin factura que las respalde: 94 por 856,640 (0.69% de las notas crédito)
por mes: {'2026-01': 17, '2026-02': 28, '2026-03': 20, '2026-04': 1, '2026-05': 1, '2026-06': 27}
clientes que cierran el período con saldo negativo: 40


Conclución el porque de los valores negativos son Notas Credito de meses anteriores. para no afectar el calculo y tener el valor real por mes correlacionamos la factura a la nota credito. y marcamos como atipico el 0.69% ya que no sabemos de donde sale o si es una factura previa a 1 Enero 2026.

## Integridad de los datos:
¿hay datos huerfanos? ventas sin cliente, o clientes sin asesor? ...

In [ ]:
maestros = {hoja: pd.read_excel(ARCHIVO, sheet_name=hoja, dtype=str)
            for hoja in ["Clientes", "Materiales", "Asesores", "Sedes"]}
ventas["material"] = ventas["Cod Material"].map(to_int)

clientes = set(maestros["Clientes"]["Cod Cliente"].map(to_int))
materiales = set(maestros["Materiales"]["Cod Material"].map(to_int))
asignaciones = maestros["Asesores"].drop_duplicates()
asignada = dict(zip(asignaciones["Cod Cliente"].map(to_int),
                    asignaciones["Cod Asesor"].str.strip()))
asesores = set(maestros["Sedes"]["Cod Asesor"].str.strip())

fallas = {
    "ventas sin cliente": int((~ventas["cliente"].isin(clientes)).sum()),
    "ventas sin material": int((~ventas["material"].isin(materiales)).sum()),
    "ventas sin asesor": int((~ventas["cliente"].isin(set(asignada))).sum()),
    "clientes sin asesor": len(clientes - set(asignada)),
    "asesores no catalogados": len(set(asignada.values()) - asesores),
    "materiales sin ventas": len(materiales - set(ventas["material"])),
    "duplicados en el grano (mes, cliente, material)":
        int(ventas.duplicated(subset=["periodo", "cliente", "material"]).sum()),
}
for etiqueta, cantidad in fallas.items():
    if cantidad:
        print(f"FALLA {etiqueta}: {cantidad}")
print("relaciones rotas:", sum(fallas.values()) or "ninguna")

print("\n asesores sin clientes:",
      sorted(asesores - set(asignada.values())))